In [1]:
%cd /content
!rm -rf RiskAware-Complaints-Engine
!git clone https://github.com/Nusultan11/RiskAware-Complaints-Engine.git
%cd /content/RiskAware-Complaints-Engine
!pip install -e .

/content
Cloning into 'RiskAware-Complaints-Engine'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 235 (delta 87), reused 205 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (235/235), 154.35 KiB | 7.35 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/RiskAware-Complaints-Engine
Obtaining file:///content/RiskAware-Complaints-Engine
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for risk-aware-complaints-engine (pyproject.toml) ... done
  Created wheel for risk-aware-complaints-engine: filename=risk_aware_complaints_engine-0.1.0-0.editable-py3-none-any.whl size=2362 sha256=a35558fc121c7f84ff0d865514424d7e26c875f26532a7f95a6b0dff8dcf22ef
  Stored in directory: /tmp/pip-ephem-wheel-

In [12]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from risk_aware.preprocessing.tfidf import tfidf_clean
import zipfile
from google.colab import files

In [13]:
train_path = Path("data/processed/train.csv")
val_path = Path("data/processed/val.csv")
test_path = Path("data/processed/test.csv")

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

In [14]:
text_col = "consumer_complaint_narrative"
target_col = "category"

for df in [train_df, val_df, test_df]:
    df.dropna(subset=[text_col, target_col], inplace=True)

x_train = train_df[text_col].astype(str).tolist()
y_train = train_df[target_col].astype(str).tolist()
x_val = val_df[text_col].astype(str).tolist()
y_val = val_df[target_col].astype(str).tolist()
x_test = test_df[text_col].astype(str).tolist()
y_test = test_df[target_col].astype(str).tolist()

In [15]:
MAX_FEATURES = 70000
MAX_DF = 0.9
C_VALUE = 3.641226167642757
CLASS_WEIGHT = "balanced"

configs = [
    {
        "name": "word_12_m3",
        "analyzer": "word",
        "ngram_range": (1, 2),
        "min_df": 3,
        "preprocessor": tfidf_clean,
        "lowercase": False,
    },
    {
        "name": "word_12_m5",
        "analyzer": "word",
        "ngram_range": (1, 2),
        "min_df": 5,
        "preprocessor": tfidf_clean,
        "lowercase": False,
    },
    {
        "name": "word_11_m3",
        "analyzer": "word",
        "ngram_range": (1, 1),
        "min_df": 3,
        "preprocessor": tfidf_clean,
        "lowercase": False,
    },
    {
        "name": "char_35_m3",
        "analyzer": "char_wb",
        "ngram_range": (3, 5),
        "min_df": 3,
        "preprocessor": None,
        "lowercase": True,
    },
]

rows = []

for cfg in configs:
    vec = TfidfVectorizer(
        analyzer=cfg["analyzer"],
        ngram_range=cfg["ngram_range"],
        min_df=cfg["min_df"],
        max_df=MAX_DF,
        max_features=MAX_FEATURES,
        preprocessor=cfg["preprocessor"],
        lowercase=cfg["lowercase"],
        sublinear_tf=True,
    )

    clf = LogisticRegression(
        C=C_VALUE,
        class_weight=CLASS_WEIGHT,
        max_iter=1000,
        n_jobs=1,
    )

    pipe = Pipeline([("tfidf", vec), ("clf", clf)])
    pipe.fit(x_train, y_train)

    pred_val = pipe.predict(x_val)
    pred_test = pipe.predict(x_test)

    rows.append({
        "config": cfg["name"],
        "val_accuracy": accuracy_score(y_val, pred_val),
        "val_macro_f1": f1_score(y_val, pred_val, average="macro"),
        "val_weighted_f1": f1_score(y_val, pred_val, average="weighted"),
        "test_accuracy": accuracy_score(y_test, pred_test),
        "test_macro_f1": f1_score(y_test, pred_test, average="macro"),
        "test_weighted_f1": f1_score(y_test, pred_test, average="weighted"),
    })

results = pd.DataFrame(rows).sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
display(results)

print("Best by test_macro_f1:", results.iloc[0]["config"])
print("Best by test_weighted_f1:", results.sort_values("test_weighted_f1", ascending=False).iloc[0]["config"])
print("Test classes:", pd.Series(y_test).nunique())

,config,val_accuracy,val_macro_f1,val_weighted_f1,test_accuracy,test_macro_f1,test_weighted_f1
0,word_11_m3,0.510386,0.304168,0.520560,0.502134,0.285760,0.511639
1,char_35_m3,0.508976,0.305247,0.520362,0.502434,0.282604,0.513603
2,word_12_m5,0.539054,0.300713,0.539687,0.527518,0.280207,0.528086
3,word_12_m3,0.539994,0.299708,0.540352,0.527293,0.280042,0.527807


Best by test_macro_f1: word_11_m3
Best by test_weighted_f1: word_12_m5
Test classes: 87


In [16]:
!PYTHONPATH=src python src/risk_aware/data/prepare.py
!PYTHONPATH=src python scripts/train_category.py
!PYTHONPATH=src python scripts/evaluate_category.py

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
Category training completed.
Macro-F1: 0.2997
Accuracy: 0.5400
Artifacts saved to: artifacts/category
Validation metrics saved to: reports/metrics/category_metrics_val.json
category_macro_f1: 0.280042
Saved metrics to: reports/metrics/category_metrics_test.json


In [17]:
bundle_name = "tfidf_baseline_v_current_bundle.zip"
paths_to_pack = [
    "artifacts/category",
    "reports/metrics/category_metrics_val.json",
    "reports/metrics/category_metrics_test.json",
    "data/processed/train.csv",
    "data/processed/val.csv",
    "data/processed/test.csv",
    "data/processed/metadata.json",
    "configs/category.yaml",
]

root = Path("/content/RiskAware-Complaints-Engine")
bundle_path = root / bundle_name

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in paths_to_pack:
        p = root / rel
        if p.is_file():
            z.write(p, arcname=rel)
        elif p.is_dir():
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f.relative_to(root)))

print("Created:", bundle_path)

Created: /content/RiskAware-Complaints-Engine/tfidf_baseline_v_current_bundle.zip


In [18]:
files.download("/content/RiskAware-Complaints-Engine/tfidf_baseline_v_current_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>